# Exploring Sample Data
Exploring all posts with >=1 comment over time period spanning January 1st, 2026 to November 1st, 2026.

In [1]:
import pandas as pd

In [6]:
sample_posts_path = "data/initial_eda/subreddit_resumes_posts.parquet"
posts = pd.read_parquet(sample_posts_path)

## Validating Data
Validating the data is what we expect by investigating distributions of values.

In [11]:
# validating the time period was respected
posts["created_utc"] = pd.to_datetime(posts["created_utc"], unit="s")
posts["created_utc"].describe()

count                         20827
mean     2026-05-14 14:14:26.002256
min             2026-01-01 00:29:21
25%      2026-02-25 15:13:13.500000
50%             2026-05-04 21:39:57
75%             2026-07-28 12:10:49
max             2026-10-31 22:55:34
Name: created_utc, dtype: object

In [16]:
posts["score"].describe()

count    20827.000000
mean         1.948384
std         14.389836
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max       1175.000000
Name: score, dtype: float64

In [32]:
# min comments is 1
posts["num_comments"].describe()

count    20827.000000
mean         2.733231
std          5.814029
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        271.000000
Name: num_comments, dtype: float64

In [19]:
posts["id"].nunique()

20827

In [24]:
# for the subreddit r/resumes
posts["subreddit"].nunique()

1

In [27]:
posts["url"].value_counts()

url
                                                                                                      410
https://ibb.co/FkH9bW5S                                                                                 4
/r/CVwriting/comments/1vdv2du/final_year_psychology_student_rate_my_cv/                                 3
/r/EngineeringResumes/comments/1vayoet/2_yoe_international_ms_grad_resume_review/                       3
https://i.redd.it/tbz1om0o95gh1.png                                                                     3
                                                                                                     ... 
https://www.reddit.com/r/resumes/comments/1q47e05/1_yoe_qos_monitoring_and_drive_test_analyst_ran/      1
https://www.reddit.com/r/resumes/comments/1q47d2u/30_yoe_unemployed_supply_chainlogistics_united/       1
https://www.reddit.com/r/resumes/comments/1q47bei/1_yoe_grad_student_ran_engineer_kpi_performance/      1
https://i.redd.it/dexw9s8rhfbg1.png       

In [ ]:
posts["url"].value_counts()

selftext
[removed]                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  9071
                                                                                                               

In [31]:
posts["title"].value_counts()

title
Roast my resume                                                                                      84
[ Removed by moderator ]                                                                             34
Resume Review                                                                                        34
Resume help                                                                                          31
Review my resume                                                                                     28
                                                                                                     ..
[1 YoE, QoS Monitoring and Drive Test Analyst , RAN Performance Optimization Engineer , Tunisia ]     1
30 YoE, Unemployed, Supply Chain/Logistics, United States                                             1
[1 YoE, Grad Student/ RAN Engineer / KPI Performance & Optimization , Tunisia]                        1
Resume Review for Software Engineering - Business Hybrid  

## Filtering Data
We need to filter the data to ensure we only consider posts with relevant data. For relevant posts, the following must apply:
- Title must be of form **[X YoE, Current Role/Unemployed, Target Role, Country]**.
    - This is expected format according to r/resumes guidelines for submitting a resume review request.
- There must be text that was not removed, deleted, or otherwise unreadable.
- The url must contain a image of the resume (not anything else)


### Filter Text Column
- removed useless posts which contain the following: "removed", "deleted", or empty content.
- leaves 9658 of the original 20827 remaining (46.37% remaining).

In [42]:
text_len = posts["selftext"].str.len()

In [50]:
text_len_75, text_len_25 = text_len.quantile(0.75), text_len.quantile(0.25)
text_len_iqr = text_len_75 - text_len_25
text_len_lower, text_len_upper = text_len_25 - 1.5 * text_len_iqr, text_len_75 + 1.5 * text_len_iqr
text_requires_review = text_len <= text_len_upper

In [53]:
invalid_words = ["[removed]", "[deleted]"]
is_text_valid = (posts["selftext"].str.len() > 0) & (posts["selftext"].isin(invalid_words) == False)

In [55]:
is_text_valid.value_counts()

selftext
False    11169
True      9658
Name: count, dtype: int64

In [56]:
is_text_valid.shape

(20827,)

In [57]:
is_text_valid.value_counts() / is_text_valid.shape[0]

selftext
False    0.536275
True     0.463725
Name: count, dtype: float64

### Filter URL Column
- only included posts with ending "jpeg" and "png"
- leaves 9215 of the original 20827 remaining (44.24%).

In [63]:
url_endings = posts["url"].str.split(".").str[-1]

In [77]:
url_ending_len = url_endings.str.len()
url_ending_len_75, url_ending_len_25 = url_ending_len.quantile(0.75), url_ending_len.quantile(0.25)
url_ending_len_iqr = url_ending_len_75 - url_ending_len_25
url_ending_lower, url_ending_upper = url_ending_len_25 - 1.5 * url_ending_len_iqr, url_ending_len_75 + 1.5 * url_ending_len_iqr
is_likely_url = url_ending_len < url_ending_len_75

In [86]:
valid_endings = ["png", "jpeg"]
url_ending = posts["url"].str.split(".").str[-1]
is_url_valid = url_ending.isin(valid_endings) == True

In [87]:
is_url_valid.value_counts()

url
False    11612
True      9215
Name: count, dtype: int64

In [88]:
is_url_valid.value_counts() / is_url_valid.shape[0]

url
False    0.557545
True     0.442455
Name: count, dtype: float64

### Filter Title Column

In [90]:
posts["title"]

0        [1 YoE, Account Executive/Customer Success Hyb...
1        [1-2 YOE, Account Executive/Customer Success H...
2        I spent months building a complete job search ...
3        [4 YoE, Sub Teacher, Remote/USA, Looking to Pi...
4        [3 YoE, Art Director/Laid Off, 2D artist/conce...
                               ...                        
20822    [1 YoE, QoS Monitoring and Drive Test Analyst ...
20823    30 YoE, Unemployed, Supply Chain/Logistics, Un...
20824    [1 YoE, Grad Student/ RAN Engineer / KPI Perfo...
20825    Resume Review for Software Engineering - Busin...
20826    [1 YoE, IT Business Analyst Intern, Associate/...
Name: title, Length: 20827, dtype: str

In [ ]:
# [X YoE, Current Role/Unemployed, Target Role, Country]
relevant_title_regex = 